# TNG × Faraday rotation: theoretical fiducial figures

観測beam・雑音・photo-z誤差・観測sample選択を入れず、TNGの理論的検証に特化した基本図。
主解析からは `run_configuration.json` のcosmology・近似cone geometryだけを読み、
元のfull gas snapshotsと全group catalogsからRMと銀河を**独立に再投影**する。
主解析の平滑化されたmapや低解像度のmapは使用しない。

すべてのgas chunks/cellsをstreamingで読み、既定でstar-forming gasも含む。
銀河はSubhaloFlagが有効で星を持つ全subhaloを用い、恒星質量閾値やランダム間引きを設けない。
snapshot図は既定で箱全体を投影。$X=R^2-\langle R^2\rangle$ を**filterなし**で計算する。

ただしセル中心NGP格子・線形補間・nearest-snapshot coneという**数値的近似**は残る。
「全セル使用」は個々のVoronoi形状を解像した厳密ray tracingと同義ではない。
`RM_GRID_N` と `THEORY_MAP_N` を変えて収束を確認する。既定はともに512。
細かい格子でも解像度の低いTNG run自体の分解能を超える情報は増えない。

PDF/300 dpi PNGと数値CSV/NPZを `fr_basic_figures/`、中間cacheを `cache/` に保存。
API key・downloadは不要。依存: numpy, scipy, pandas, matplotlib, h5py, astropy。

In [ ]:
from pathlib import Path
import json, hashlib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.patches import Polygon
import h5py
from astropy.cosmology import FlatLambdaCDM
from IPython.display import display

## 1. 設定

`MAIN_OUTPUT_DIR` はcone geometryの設定JSONの保存先。TNG mountが変わった場合は `BASE_PATH` を指定。
主解析と同じTNG run・赤方偏移範囲・視野を使うが、銀河sampleと数値格子は独立に指定する。
既定で全gasと全star-bearing valid subhalosを使用する。

`RM_GRID_N=512` の3成分float64格子はdisk上約3 GiB/snapshot。
格子はmemory-mapped cacheで構築し、snapshotごとに投影して解放する。
大きいrunの初回は全snapshotのgas読込が必要。cache完成後は再読込しない。
`GAS_SLAB_DEPTH_CMPC=None` は全箱投影。値を指定した場合だけ幾何学的slabへ限定する。

In [ ]:
NOTEBOOK_DIR = (Path.cwd()/'detect-mag'
                if (Path.cwd()/'detect-mag'/'tng_fr_basic_figures.ipynb').is_file()
                else Path.cwd())
MAIN_OUTPUT_DIR = NOTEBOOK_DIR/'fr_photoz_outputs'
OUTPUT_DIR = NOTEBOOK_DIR/'fr_basic_figures'
CACHE_DIR = NOTEBOOK_DIR/'cache'
BASE_PATH = None             # override simulation mount; default: main run_configuration
MAKE_GAS_FIGURES = True      # False: cone/maps/spectra only, no gas snapshot access
THEORY_MAP_N = 512           # numerical angular grid, not an observational beam
RM_GRID_N = 512              # numerical gas grid; compare 256/512/1024 for convergence
GALAXY_MIN_STELLAR_MASS_MSUN = 0.0
GALAXY_MIN_STAR_PARTICLES = 1 # all valid star-bearing subhalos; not a completeness cut
RM_USE_CELL_HYDROGEN_FRACTION = True
RM_EXCLUDE_STAR_FORMING = False
LOS_STEP_IN_GRID_CELLS = 0.5
SPECTRUM_N_BANDS = 15
GAS_SNAPSHOT = 33            # full snapshot, z ~ 2; independent of cone target bins
GAS_MAP_N = 512
GAS_SLAB_AXIS = 2
GAS_SLAB_CENTER_FRACTION = 0.5
GAS_SLAB_DEPTH_CMPC = None
GAS_READ_BATCH = 250_000
GAS_EXCLUDE_STAR_FORMING = False  # sensitivity option; metadata recorded
PHASE_LOG_NH_EDGES = np.linspace(-8, 4, 121)
PHASE_LOG_B_EDGES = np.linspace(-12, 3, 121) # physical microGauss
DPI = 300
assert GAS_SLAB_AXIS in (0,1,2) and GAS_MAP_N >= 8
assert 0 <= GAS_SLAB_CENTER_FRACTION < 1
assert GAS_SLAB_DEPTH_CMPC is None or GAS_SLAB_DEPTH_CMPC > 0
assert THEORY_MAP_N >= 16 and RM_GRID_N >= 8 and LOS_STEP_IN_GRID_CELLS > 0
assert GALAXY_MIN_STELLAR_MASS_MSUN >= 0 and GALAXY_MIN_STAR_PARTICLES >= 1
for p in (OUTPUT_DIR,CACHE_DIR): p.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':11,'axes.labelsize':12,'savefig.dpi':DPI,
                     'figure.dpi':110,'font.family':'DejaVu Sans','pdf.fonttype':42})
def save_figure(fig,name):
    fig.savefig(OUTPUT_DIR/(name+'.pdf'),bbox_inches='tight')
    fig.savefig(OUTPUT_DIR/(name+'.png'),dpi=DPI,bbox_inches='tight')
    plt.show()
def digest_file(path):
    d=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): d.update(b)
    return d.hexdigest()

## 2. Cone geometryを読む

主解析から読み込むのは設定JSONのみ。元のgas/group catalogを使い、
より細かい指定格子で同じ幾何学へ再投影する。主解析のmass selectionは引き継がない。
coneは有限のz範囲だけで、CMB最終散乱面までの全磁場を含むものではない。

In [ ]:
path=MAIN_OUTPUT_DIR/'run_configuration.json'
if not path.is_file(): raise FileNotFoundError(f'{path}: run the main notebook first.')
config=json.loads(path.read_text())
if config['MODE']!='tng': raise ValueError('This theoretical notebook requires a TNG configuration.')
for key in ['tiles','omega_m','box_mpc','h','TRUE_Z_EDGES','CONE_Z_RANGE']:
    if key not in config: raise KeyError(f'Missing {key}: save configuration with the current main notebook.')
target_edges=np.asarray(config['TRUE_Z_EDGES'],float)
layer_edges=np.r_[config['CONE_Z_RANGE'][0],target_edges,config['CONE_Z_RANGE'][1]]
assert np.all(np.diff(layer_edges)>0)
n_target=len(target_edges)-1
n=int(THEORY_MAP_N);grid_n=int(RM_GRID_N);fov=float(config['FOV_DEG'])
h=float(config['h']);box=float(config['box_mpc'])
cosmo=FlatLambdaCDM(H0=100*h,Om0=float(config['omega_m']))
theta=np.deg2rad(fov);pix_sr=(theta/n)**2;npix=n*n
label_prefix=''
input_hashes={'run_configuration.json':digest_file(path)}
print('Geometry source:',MAIN_OUTPUT_DIR,'; simulation:',config['SIMULATION'])
print('Actual patch area:',fov*fov,'deg²; no beam, noise, thinning or area extrapolation.')
print('Numerical grids: gas',grid_n,'angular',n)

## 3. Snapshotガス・磁場分布（streaming + cache）

$B_{\mathrm{phys}}=B_{\mathrm{code}}(h/a^2)\,2.60\times10^{-6}$ G。
ガス質量とDensityの単位は [TNG仕様](https://www.tng-project.org/data/docs/specifications/) に従う。
ガス図はセル中心NGPで箱全体（指定時のみslab）へ投影。磁場図は同じ投影範囲の**質量重み付き**rms強度と平均LOS成分であり、
体積平均磁場でも、RMでもない。

$$
\Sigma_g=\frac{\sum M_g}{A_{\mathrm{physical}}},\qquad
B_{\mathrm{rms},M}=\sqrt{\frac{\sum M_g|\boldsymbol B|^2}{\sum M_g}},\qquad
\overline{B}_{\parallel,M}=\frac{\sum M_g B_\parallel}{\sum M_g}.
$$
密度–磁場分布は**snapshot全体**の質量重み付き分布。固定 $X_H=0.76$ で
$n_H=X_H\rho_{\mathrm{phys}}/m_p$ を定義し、electron densityとは区別する。
star-forming gasは既定で含む。密度/磁場がゼロのセルと表示範囲外の質量は別に集計する。

In [ ]:
MSUN_G=1.98847e33; KPC_CM=3.0856775814913673e21; MP_G=1.67262192369e-24
GAS_CACHE_VERSION='fiducial_slab_mass_B_v1'
def snapshot_files(base,snap):
    return sorted((base/f'snapdir_{snap:03d}').glob(f'snap_{snap:03d}.*.hdf5'),
                  key=lambda p:int(p.name.split('.')[-2]))
def find_gas_base():
    if BASE_PATH is not None: return Path(BASE_PATH).expanduser().resolve()
    candidates=[Path(config['base_path'])]
    for root in [NOTEBOOK_DIR,*list(NOTEBOOK_DIR.parents)[:4],Path.home()]:
        candidates += [root/'sims.TNG'/config['SIMULATION']/'output']
    for p in candidates:
        if (p/f'snapdir_{GAS_SNAPSHOT:03d}').is_dir(): return p.resolve()
    raise FileNotFoundError('Gas snapshot mount not found; set BASE_PATH or MAKE_GAS_FIGURES=False.')

def build_gas_products():
    base=find_gas_base(); files=snapshot_files(base,GAS_SNAPSHOT)
    if not files: raise FileNotFoundError(f'No full snapshot {GAS_SNAPSHOT} at {base}')
    with h5py.File(files[0],'r') as f: header=dict(f['Header'].attrs)
    if len(files)!=int(header['NumFilesPerSnapshot']): raise ValueError('Incomplete snapshot chunks')
    gh=float(header['HubbleParam']); a=float(header['Time']); z=float(header['Redshift'])
    gas_box=float(header['BoxSize'])/(1000*gh)
    if not np.isclose(gas_box,box) or not np.isclose(gh,h):
        raise ValueError('Gas snapshot cosmology/box differs from main cone.')
    assert GAS_SLAB_DEPTH_CMPC is None or GAS_SLAB_DEPTH_CMPC<=gas_box
    assert np.isclose(a,1/(1+z))
    meta=dict(version=GAS_CACHE_VERSION,base=str(base),simulation=config['SIMULATION'],
        snap=GAS_SNAPSHOT,n=GAS_MAP_N,axis=GAS_SLAB_AXIS,center=GAS_SLAB_CENTER_FRACTION,
        depth=GAS_SLAB_DEPTH_CMPC,exclude_SF=GAS_EXCLUDE_STAR_FORMING,
        nh_edges=PHASE_LOG_NH_EDGES.tolist(),b_edges=PHASE_LOG_B_EDGES.tolist(),XH=0.76,
        files=[(p.name,p.stat().st_size,p.stat().st_mtime_ns) for p in files])
    key=hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()[:20]
    cache=CACHE_DIR/f'gas_basic_figures_{key}.npz'
    if cache.exists():
        with np.load(cache,allow_pickle=False) as f: result={k:f[k].copy() for k in f.files}
        print('Gas cache hit:',cache.name)
        return result,meta,cache
    mass_map=np.zeros((GAS_MAP_N,GAS_MAP_N)); mb2=mass_map.copy(); mbpar=mass_map.copy()
    phase=np.zeros((len(PHASE_LOG_NH_EDGES)-1,len(PHASE_LOG_B_EDGES)-1))
    totals=np.zeros(4) # selected gas mass, phase-covered mass, invalid/zero nH or B mass, slab mass
    transverse=[k for k in range(3) if k!=GAS_SLAB_AXIS]
    start_time=time.time()
    for file_index,file in enumerate(files):
        with h5py.File(file,'r') as f:
            if 'PartType0' not in f: continue
            g=f['PartType0']
            for name in ['Coordinates','Masses','Density','MagneticField']:
                if name not in g: raise KeyError(f'{file}: missing {name}; full gas snapshot required')
            if GAS_EXCLUDE_STAR_FORMING and 'StarFormationRate' not in g: raise KeyError('Missing SFR')
            for start in range(0,len(g['Masses']),GAS_READ_BATCH):
                sl=slice(start,start+GAS_READ_BATCH)
                pos=g['Coordinates'][sl].astype(float)/(1000*gh)
                mass=g['Masses'][sl].astype(float)*1e10/gh
                B=g['MagneticField'][sl].astype(float)*(gh/a**2)*2.60 # microGauss
                rho=g['Density'][sl].astype(float)*(1e10*MSUN_G/gh)/(a*KPC_CM/gh)**3
                good=np.isfinite(pos).all(1)&np.isfinite(mass)&(mass>0)&np.isfinite(B).all(1)
                if GAS_EXCLUDE_STAR_FORMING: good &= g['StarFormationRate'][sl]<=0
                pos,mass,B,rho=pos[good],mass[good],B[good],rho[good]
                bmag=np.linalg.norm(B,axis=1); nh=0.76*rho/MP_G
                totals[0]+=mass.sum()
                valid=(nh>0)&np.isfinite(nh)&(bmag>0)
                totals[2]+=mass[~valid].sum()
                hist=np.histogram2d(np.log10(nh[valid]),np.log10(bmag[valid]),
                    bins=[PHASE_LOG_NH_EDGES,PHASE_LOG_B_EDGES],weights=mass[valid])[0]
                phase+=hist; totals[1]+=hist.sum()
                distance=(pos[:,GAS_SLAB_AXIS]-GAS_SLAB_CENTER_FRACTION*gas_box+gas_box/2)%gas_box-gas_box/2
                keep=np.ones(len(mass),dtype=bool) if GAS_SLAB_DEPTH_CMPC is None else abs(distance)<GAS_SLAB_DEPTH_CMPC/2
                xy=pos[keep][:,transverse]%gas_box
                idx=np.floor(xy/gas_box*GAS_MAP_N).astype(int)
                flat=idx[:,0]*GAS_MAP_N+idx[:,1]
                weights=[mass[keep],mass[keep]*bmag[keep]**2,mass[keep]*B[keep,GAS_SLAB_AXIS]]
                for out,w in zip([mass_map,mb2,mbpar],weights):
                    out+=np.bincount(flat,weights=w,minlength=GAS_MAP_N**2).reshape(out.shape)
                totals[3]+=mass[keep].sum()
        print(f'Gas chunks {file_index+1}/{len(files)}; elapsed {(time.time()-start_time)/60:.1f} min')
    if mass_map.sum()<=0 or totals[0]<=0: raise ValueError('No usable gas in snapshot/slab')
    assert np.isclose(mass_map.sum(),totals[3])
    result=dict(mass=mass_map,mb2=mb2,mbpar=mbpar,phase=phase,totals=totals,
                a=np.array(a),z=np.array(z),box=np.array(gas_box))
    # Replace only a completed temporary cache; an interrupted run never becomes a cache hit.
    temp=cache.with_suffix('.tmp.npz');np.savez_compressed(temp,**result);temp.replace(cache)
    return result,meta,cache

gas_metadata=None; gas_cache=None
if MAKE_GAS_FIGURES:
    if config['MODE']!='tng': raise ValueError('Synthetic cone has no physical gas snapshot.')
    gas,gas_metadata,gas_cache=build_gas_products()
    a=float(gas['a']); m=gas['mass']; extent=[0,float(gas['box'])]*2
    area_pc2=(float(gas['box'])/GAS_MAP_N*a*1e6)**2
    sigma=m/area_pc2
    brms=np.sqrt(np.divide(gas['mb2'],m,out=np.full_like(m,np.nan),where=m>0))
    bpar=np.divide(gas['mbpar'],m,out=np.full_like(m,np.nan),where=m>0)
    fig,axes=plt.subplots(1,3,figsize=(15,4.6),constrained_layout=True)
    for ax,field,title,unit,cmap in zip(axes,[sigma,brms,bpar],
        [r'Gas surface density',r'Mass-weighted $B_{\mathrm{rms}}$',r'Mass-weighted $B_\parallel$'],
        [r'$M_\odot\,\mathrm{pc}^{-2}$',r'$\mu\mathrm{G}$',r'$\mu\mathrm{G}$'],['magma','viridis','RdBu_r']):
        if field is bpar:
            vmax=max(float(np.nanpercentile(abs(field),99)),1e-20)
            norm=SymLogNorm(linthresh=vmax*0.01,vmin=-vmax,vmax=vmax)
        else:
            positive=field[np.isfinite(field)&(field>0)]
            if not len(positive): raise ValueError('No positive values for log gas/B map')
            lo,hi=np.percentile(positive,[1,99]);hi=max(hi,lo*1.01)
            norm=LogNorm(vmin=lo,vmax=hi)
        im=ax.imshow(np.ma.masked_invalid(np.where(m>0,field,np.nan)).T,origin='lower',
                     extent=extent,cmap=cmap,norm=norm,rasterized=True)
        ax.set(title=title,xlabel='Transverse distance [cMpc]',ylabel='Transverse distance [cMpc]')
        fig.colorbar(im,ax=ax,label=unit,shrink=0.82)
    projection_label='full box' if GAS_SLAB_DEPTH_CMPC is None else f'{GAS_SLAB_DEPTH_CMPC:g} cMpc slab'
    fig.suptitle(f"{config['SIMULATION']}, z={float(gas['z']):.3f}; {projection_label}; NGP")
    save_figure(fig,'01_gas_magnetic_projection')
    dnh=np.diff(PHASE_LOG_NH_EDGES); db=np.diff(PHASE_LOG_B_EDGES)
    pdf=gas['phase']/gas['totals'][0]/(dnh[:,None]*db[None,:])
    positive=pdf[pdf>0]
    if not len(positive): raise ValueError('Density/B plotting range contains no mass; widen edges.')
    fig,ax=plt.subplots(figsize=(6.7,5.0),constrained_layout=True)
    im=ax.pcolormesh(PHASE_LOG_NH_EDGES,PHASE_LOG_B_EDGES,np.ma.masked_less_equal(pdf.T,0),
        norm=LogNorm(vmin=max(positive.min(),positive.max()*1e-6),vmax=positive.max()),
        cmap='magma',rasterized=True)
    ax.set(xlabel=r'$\log_{10}(n_H/\mathrm{cm}^{-3})$',ylabel=r'$\log_{10}(|B|/\mu\mathrm{G})$',
           title=f"{config['SIMULATION']}, z={float(gas['z']):.3f}; full snapshot")
    fig.colorbar(im,ax=ax,label='Gas mass fraction per dex²')
    save_figure(fig,'02_density_magnetic_distribution')
    print('Phase plot mass coverage:',gas['totals'][1]/gas['totals'][0])
    print('Invalid/zero nH or B mass fraction:',gas['totals'][2]/gas['totals'][0])
    np.savez_compressed(OUTPUT_DIR/'gas_figure_data.npz',**gas,nh_edges=PHASE_LOG_NH_EDGES,b_edges=PHASE_LOG_B_EDGES)

### 3b. 元データから理論coneを再構築（cacheあり）

電子数重み付き磁場を全gasセルから格子にdepositし、同じconeへ視線積分する。
質量閾値なしの銀河catalogも元データから読み直す。beamは加えない。


In [ ]:
from scipy.ndimage import map_coordinates

def group_files(base,snap):
    return sorted((base/f'groups_{snap:03d}').glob(f'fof_subhalo_tab_{snap:03d}.*.hdf5'),
                  key=lambda p:int(p.name.split('.')[-2]))
def file_manifest(files):
    return [(p.name,p.stat().st_size,p.stat().st_mtime_ns) for p in files]
def key_for(meta):
    return hashlib.sha256(json.dumps(meta,sort_keys=True).encode()).hexdigest()[:20]
def validate_chunks(files,kind):
    if not files: raise FileNotFoundError(f'No {kind} files')
    with h5py.File(files[0],'r') as f:
        hd=dict(f['Header'].attrs)
    count=int(hd['NumFilesPerSnapshot'] if kind=='gas' else hd['NumFiles'])
    if len(files)!=count: raise FileNotFoundError(f'Incomplete {kind} chunks: {len(files)}/{count}')
    if not np.isclose(float(hd['BoxSize'])/(1000*float(hd['HubbleParam'])),box):
        raise ValueError('Box differs from geometry')
    if not np.isclose(float(hd['HubbleParam']),h): raise ValueError('Cosmology differs from geometry')
    return hd

def build_electron_B_grid(base,snap,files):
    hd=validate_chunks(files,'gas');z=float(hd['Redshift']);a=1/(1+z)
    meta=dict(version='theory_electron_B_NGP_v1',base=str(base),snap=int(snap),grid=grid_n,
              cell_XH=RM_USE_CELL_HYDROGEN_FRACTION,exclude_SF=RM_EXCLUDE_STAR_FORMING,
              files=file_manifest(files),B_conversion=2.60e-6)
    cache=CACHE_DIR/f'theory_rm_grid_{key_for(meta)}.npy'
    if cache.exists():
        q=np.load(cache,mmap_mode='r',allow_pickle=False)
        if q.shape!=(3,grid_n,grid_n,grid_n): raise ValueError('Wrong cached grid shape')
        print('RM grid cache:',cache.name)
        return q
    temporary=cache.with_suffix('.tmp.npy')
    q=np.lib.format.open_memmap(temporary,mode='w+',dtype=np.float64,
                              shape=(3,grid_n,grid_n,grid_n))
    q[:]=0
    volume_cm3=(box/grid_n*1000*KPC_CM)**3
    included=read_count=0
    for file_index,file in enumerate(files):
        with h5py.File(file,'r') as f:
            if 'PartType0' not in f: continue
            gas=f['PartType0']
            for name in ['Coordinates','Masses','ElectronAbundance','MagneticField']:
                if name not in gas: raise KeyError(f'{file}: missing {name}')
            if RM_USE_CELL_HYDROGEN_FRACTION and 'GFM_Metals' not in gas:
                raise KeyError('GFM_Metals required for cell hydrogen fractions; disable option for fixed XH.')
            if RM_EXCLUDE_STAR_FORMING and 'StarFormationRate' not in gas: raise KeyError('Missing SFR')
            for start in range(0,len(gas['Masses']),GAS_READ_BATCH):
                sl=slice(start,start+GAS_READ_BATCH)
                pos=gas['Coordinates'][sl].astype(float)/(1000*h)
                mass=gas['Masses'][sl].astype(float)*1e10*MSUN_G/h
                xe=gas['ElectronAbundance'][sl].astype(float)
                xh=gas['GFM_Metals'][sl,0].astype(float) if RM_USE_CELL_HYDROGEN_FRACTION else np.full(len(mass),0.76)
                b=gas['MagneticField'][sl].astype(float)*(h/a**2)*2.60
                good=np.isfinite(pos).all(1)&np.isfinite(b).all(1)&np.isfinite(mass)&(mass>0)
                good &= np.isfinite(xe)&(xe>=0)&np.isfinite(xh)&(xh>0)&(xh<=1)
                if RM_EXCLUDE_STAR_FORMING:good &= gas['StarFormationRate'][sl]<=0
                read_count+=len(mass);included+=int(good.sum())
                ijk=np.floor((pos[good]%box)/box*grid_n).astype(np.int64)
                idx=(ijk[:,0]*grid_n+ijk[:,1])*grid_n+ijk[:,2]
                ne=mass[good]*xe[good]*xh[good]/MP_G/volume_cm3
                for k in range(3):np.add.at(q[k].reshape(-1),idx,ne*b[good,k])
        print(f'RM snap {snap}: gas chunks {file_index+1}/{len(files)}')
    if included==0:raise ValueError('No usable gas cells')
    q.flush();del q;temporary.replace(cache)
    print(f'RM snap {snap}: {included:,}/{read_count:,} cells used; no subsampling')
    return np.load(cache,mmap_mode='r',allow_pickle=False)

def read_all_galaxies(files):
    validate_chunks(files,'group');parts=[];global_id=0
    for file in files:
        with h5py.File(file,'r') as f:
            if 'Subhalo' not in f:continue
            g=f['Subhalo'];count=len(g['SubhaloPos'])
            mass=g['SubhaloMassType'][:,4].astype(float)*1e10/h
            stars=g['SubhaloLenType'][:,4]
            good=(mass>0)&(mass>=GALAXY_MIN_STELLAR_MASS_MSUN)&(stars>=GALAXY_MIN_STAR_PARTICLES)
            if 'SubhaloFlag' not in g:raise KeyError('SubhaloFlag required to identify valid galaxies')
            good &= g['SubhaloFlag'][:].astype(bool)
            if good.any():parts.append(dict(pos=g['SubhaloPos'][:][good].astype(float)/(1000*h),
                mass=mass[good],subhalo_id=np.arange(global_id,global_id+count)[good]))
            global_id+=count
    if not parts:raise ValueError('No star-bearing valid subhalos')
    return {key:np.concatenate([p[key] for p in parts]) for key in parts[0]}

z_lookup=np.linspace(0,max(6.,layer_edges[-1]+1),20001)
chi_lookup=cosmo.comoving_distance(z_lookup).value
z_to_chi=lambda z:np.interp(z,z_lookup,chi_lookup)
chi_to_z=lambda chi:np.interp(chi,chi_lookup,z_lookup)
layer_chi=z_to_chi(layer_edges)
axis=(np.arange(n)+0.5)/n*theta-theta/2
AX,AY=np.meshgrid(axis,axis,indexing='ij')

def project_rm(q,tile):
    perm=np.asarray(tile['perm'],int);signs=np.asarray(tile['signs']);offset=np.asarray(tile['offset'])
    lo0,hi0=tile['chi_lo'],tile['chi_hi']
    cuts=np.unique(np.r_[lo0,hi0,layer_chi[(layer_chi>lo0)&(layer_chi<hi0)]])
    result=np.zeros((len(layer_edges)-1,n,n))
    for lo,hi in zip(cuts[:-1],cuts[1:]):
        steps=max(1,int(np.ceil((hi-lo)/(LOS_STEP_IN_GRID_CELLS*box/grid_n))))
        step=(hi-lo)/steps
        layer=np.searchsorted(layer_chi,(lo+hi)/2,side='right')-1
        for chi in lo+(np.arange(steps)+0.5)*step:
            global_pos=np.array([AX*chi,AY*chi,np.full_like(AX,chi-tile['chi_mid'])])
            original=np.empty_like(global_pos)
            for k in range(3):original[perm[k]]=offset[perm[k]]+signs[k]*global_pos[k]
            coords=original/box*grid_n-0.5
            value=signs[2]*map_coordinates(q[perm[2]],coords,order=1,mode='grid-wrap',prefilter=False)
            result[layer]+=0.812*value*(step*1e6)
    return result

def project_galaxies(parent,tile):
    perm=np.asarray(tile['perm'],int);signs=np.asarray(tile['signs']);offset=np.asarray(tile['offset'])
    relative=(parent['pos']-offset+box/2)%box-box/2
    local=relative[:,perm]*signs
    chi=tile['chi_mid']+local[:,2]
    angles=local[:,:2]/chi[:,None]
    good=(chi>=tile['chi_lo'])&(chi<tile['chi_hi'])&np.all(abs(angles)<theta/2,axis=1)
    pix=np.floor((angles[good]/theta+0.5)*n).astype(int)
    return pd.DataFrame(dict(ix=pix[:,0],iy=pix[:,1],theta_x=angles[good,0],theta_y=angles[good,1],
        z_cos=chi_to_z(chi[good]),stellar_mass=parent['mass'][good],snap=tile['snap'],tile=tile['tile'],
        subhalo_id=parent['subhalo_id'][good]))

base=find_gas_base();snapshots=sorted(set(int(t['snap']) for t in config['tiles']))
gas_files={s:snapshot_files(base,s) for s in snapshots}
group_chunks={s:group_files(base,s) for s in snapshots}
for s in snapshots:
    validate_chunks(gas_files[s],'gas');validate_chunks(group_chunks[s],'group')
cone_metadata=dict(version='unfiltered_theory_cone_v1',base=str(base),tiles=config['tiles'],
    target_edges=target_edges.tolist(),layer_edges=layer_edges.tolist(),map_n=n,grid_n=grid_n,
    fov=fov,h=h,omega_m=float(config['omega_m']),los_step=LOS_STEP_IN_GRID_CELLS,
    min_mass=GALAXY_MIN_STELLAR_MASS_MSUN,min_stars=GALAXY_MIN_STAR_PARTICLES,
    cell_XH=RM_USE_CELL_HYDROGEN_FRACTION,exclude_SF=RM_EXCLUDE_STAR_FORMING,
    gas_files={str(s):file_manifest(gas_files[s]) for s in snapshots},
    group_files={str(s):file_manifest(group_chunks[s]) for s in snapshots})
cone_cache=CACHE_DIR/f'theory_cone_{key_for(cone_metadata)}.npz'
if cone_cache.exists():
    with np.load(cone_cache,allow_pickle=False) as f:
        rm_layers=f['rm'].copy();galaxies=pd.DataFrame.from_records(f['galaxies'])
    print('Theory cone cache:',cone_cache.name)
else:
    rm_layers=np.zeros((len(layer_edges)-1,n,n));pieces=[]
    for snap in snapshots:
        q=build_electron_B_grid(base,snap,gas_files[snap]);parent=read_all_galaxies(group_chunks[snap])
        for tile in [t for t in config['tiles'] if int(t['snap'])==snap]:
            rm_layers+=project_rm(q,tile);pieces.append(project_galaxies(parent,tile))
        del q,parent
    galaxies=pd.concat(pieces,ignore_index=True)
    if not len(galaxies):raise ValueError('Empty cone galaxy catalog')
    temporary=cone_cache.with_suffix('.tmp.npz')
    np.savez_compressed(temporary,rm=rm_layers,galaxies=galaxies.to_records(index=False))
    temporary.replace(cone_cache)
assert rm_layers.shape==(len(layer_edges)-1,n,n) and np.isfinite(rm_layers).all()
assert ((galaxies.z_cos>=layer_edges[0])&(galaxies.z_cos<layer_edges[-1])).all()
galaxies.to_csv(OUTPUT_DIR/'theory_cone_galaxies.csv',index=False)
np.savez_compressed(OUTPUT_DIR/'theory_cone_rm_layers.npz',rm=rm_layers,layer_edges=layer_edges)
print('Theory galaxies:',len(galaxies),'; unfiltered RM rms:',np.std(rm_layers.sum(0)),'rad/m²')

## 4. 近似lightconeと銀河の赤方偏移分布

主解析と同じtile境界・nearest snapshot割当て。元catalogの連続座標からcone内の角度を計算する。
全star-bearing valid subhalosを使い、表示でも間引かない。
箱の周期複製と回転を使う近似であり、独立な広域宇宙ではない。

In [ ]:
zgrid=np.linspace(0,layer_edges[-1]+0.2,10001)
chigrid=cosmo.comoving_distance(zgrid).value
chi=np.interp(galaxies.z_cos,zgrid,chigrid)
theta_x=galaxies.theta_x.to_numpy()
x=theta_x*chi
tiles=config['tiles']; palette=plt.get_cmap('tab20')
fig,axes=plt.subplots(1,2,figsize=(13,4.8),constrained_layout=True)
ax=axes[0]
for j,t in enumerate(tiles):
    lo,hi=t['chi_lo'],t['chi_hi']; half=theta/2
    poly=Polygon([[lo,-lo*half],[hi,-hi*half],[hi,hi*half],[lo,lo*half]],
                  facecolor=palette(j%20),alpha=0.18,edgecolor='0.6',linewidth=0.6)
    ax.add_patch(poly)
    ax.text((lo+hi)/2,hi*half+10,f"{t['snap']}",rotation=90,fontsize=7,ha='center')
sc=ax.scatter(chi,x,c=galaxies.z_cos,s=1,cmap='viridis',rasterized=True)
for z in target_edges:
    ax.axvline(np.interp(z,zgrid,chigrid),ls='--',color='0.2',lw=0.8)
ax.set(xlabel=r'LOS comoving distance $\chi$ [cMpc]',ylabel='Transverse x [cMpc]',
       title=label_prefix+'Approximate cone; labels: snapshot IDs')
ax.set_xlim(tiles[0]['chi_lo'],tiles[-1]['chi_hi'])
ax.set_ylim(-tiles[-1]['chi_hi']*theta/2-10,tiles[-1]['chi_hi']*theta/2+75)
sec=ax.secondary_xaxis('top',functions=(lambda c:np.interp(c,chigrid,zgrid),
                                      lambda z:np.interp(z,zgrid,chigrid)))
sec.set_xlabel('Cosmological redshift')
fig.colorbar(sc,ax=ax,label='True redshift',shrink=0.8)
z_edges=np.linspace(layer_edges[0],layer_edges[-1],61)
nz,_=np.histogram(galaxies.z_cos,bins=z_edges)
nz_density=nz/(fov**2*np.diff(z_edges))
axes[1].step(z_edges,np.r_[nz_density,nz_density[-1]],where='post',color='black')
for i,(lo,hi) in enumerate(zip(target_edges[:-1],target_edges[1:])):
    axes[1].axvspan(lo,hi,color=palette(i*2),alpha=0.16)
axes[1].set(xlabel='True cosmological redshift',ylabel=r'$dN/(dz\,d\Omega)$ [deg$^{-2}$]',
            title='All selected star-bearing subhalos; true z')
save_figure(fig,'03_lightcone_and_nz')
pd.DataFrame(dict(z_lo=z_edges[:-1],z_hi=z_edges[1:],count=nz,
    dN_dz_deg2=nz/(fov**2*np.diff(z_edges)))).to_csv(OUTPUT_DIR/'galaxy_nz.csv',index=False)
pd.DataFrame([{k:v for k,v in t.items() if k not in ('perm','signs','offset')} for t in tiles]).to_csv(OUTPUT_DIR/'lightcone_tiles.csv',index=False)

## 5. 理想的な銀河・RM map（平滑化なし）

真のcosmological zで分類し、$g_i=N_i/\bar N_i-1$。
表示・スペクトルの両方で同じ未平滑化count mapを使用する。疎なpixelもそのまま表示する。
RMにはbeamやGaussian filterを適用しない。符号付き層を足してから
$X=R^2-\langle R^2\rangle$ とする。細かい構造はTNG自体と数値格子の分解能に制限される。

In [ ]:
lx=2*np.pi*np.fft.fftfreq(n,d=theta/n)
LX,LY=np.meshgrid(lx,lx,indexing='ij'); LMAG=np.hypot(LX,LY)
R=rm_layers.sum(0)
X=R**2-np.mean(R**2)
gmaps=[];counts=[]
for lo,hi in zip(target_edges[:-1],target_edges[1:]):
    keep=(galaxies.z_cos>=lo)&(galaxies.z_cos<hi)
    flat=galaxies.loc[keep,'ix'].to_numpy(dtype=int)*n+galaxies.loc[keep,'iy'].to_numpy(dtype=int)
    cnt=np.bincount(flat,minlength=npix).reshape(n,n)
    if not cnt.sum(): raise ValueError(f'Empty true-z bin {lo}-{hi}')
    gmaps.append(cnt/cnt.mean()-1);counts.append(int(cnt.sum()))
gmaps=np.array(gmaps);counts=np.array(counts)
fig,axes=plt.subplots(1,n_target,figsize=(5*n_target,4.6),squeeze=False,constrained_layout=True)
for i,ax in enumerate(axes[0]):
    view=gmaps[i]
    im=ax.imshow(view.T,origin='lower',extent=[-fov/2,fov/2]*2,cmap='viridis',rasterized=True)
    ax.set(xlabel='Angle x [deg]',ylabel='Angle y [deg]',
        title=f'{target_edges[i]:g}<z<{target_edges[i+1]:g}; N={counts[i]}')
    fig.colorbar(im,ax=ax,label=r'$\delta_g$ (unfiltered)',shrink=0.8)
fig.suptitle('Fiducial galaxies; no spatial smoothing or thinning')
save_figure(fig,'04_fiducial_galaxy_maps')
fig,axes=plt.subplots(1,2,figsize=(10,4.6),constrained_layout=True)
for ax,field,title,unit in zip(axes,[R,X],['Signed RM; no filter','Centered RM²; no filter'],
                             ['rad m$^{-2}$','rad² m$^{-4}$']):
    vmax=max(float(np.max(abs(field))),1e-30)
    im=ax.imshow(field.T,origin='lower',extent=[-fov/2,fov/2]*2,cmap='RdBu_r',
        norm=SymLogNorm(linthresh=vmax*0.01,vmin=-vmax,vmax=vmax),rasterized=True)
    ax.set(title=title,xlabel='Angle x [deg]',ylabel='Angle y [deg]')
    fig.colorbar(im,ax=ax,label=unit,shrink=0.8)
fig.suptitle(f'Full cone {layer_edges[0]:g}<z<{layer_edges[-1]:g}; no beam or Gaussian filter')
save_figure(fig,'05_fiducial_rm_maps')
np.savez_compressed(OUTPUT_DIR/'fiducial_maps.npz',R=R,X=X,gmaps=gmaps,counts=counts,
                    target_edges=target_edges,layer_edges=layer_edges,fov_deg=fov,rm_fwhm_arcmin=0.0,numerical_grid_n=grid_n)

## 6. Fiducial角度パワースペクトル

主解析と同じFFT規格化を使うが、観測由来のL上限は引き継がない。
ゼロmodeは除外し、角度pixelとgas格子の数値的Nyquist上限までを表示する。
Nyquist近傍にはNGP・補間・aliasingの影響があり、格子間比較で信頼範囲を確認する。
特にRM²の積は高周波modeを作るので、L cutだけでaliasingを完全に除けるものではない。
$$
C_L^{ab}=\frac{\Omega_{\mathrm{pix}}}{N_{\mathrm{pix}}}
\operatorname{Re}(F_aF_b^*)\quad\text{（annular average）}.
$$
$C_L^{gg}$ の実線は有限catalogのshot noiseを含み、点線は $N_L^{gg}=1/\bar n_{\mathrm{sr}}$
を引いた診断値。点線が負になっても隠さない。
$C_L^{XX}$ は全coneのRM²のautoであり、各z層のautoではない。
$C_L^{Xg_i}$ は全coneのRM²と真のzで選択した銀河とのcross。
有限視野のcrossは負にもなり得るのでsymlogで描く。負を絶対値に変えない。

各bandのmode数をCSVに保存するが、単一の周期箱coneから厳密なensemble平均や誤差を主張しない。
実サーベイのmask・window coupling・connected covarianceはここでは計算しない。

In [ ]:
ell_min=2*np.pi/theta
chi_min=float(cosmo.comoving_distance(float(layer_edges[0])).value)
grid_limit=np.pi*chi_min/(box/grid_n)
ell_limit=min(np.pi*n/theta,grid_limit)
if ell_limit<=1.5*ell_min: raise ValueError('Insufficient resolved modes')
band_edges=np.geomspace(ell_min*0.99,ell_limit,int(SPECTRUM_N_BANDS)+1)
bands=[(lo,hi,(LMAG>=lo)&(LMAG<hi)) for lo,hi in zip(band_edges[:-1],band_edges[1:])]
bands=[(lo,hi,m) for lo,hi,m in bands if m.sum()>=4]
ell=np.array([LMAG[m].mean() for lo,hi,m in bands])
def spectrum(a,b):
    power=pix_sr/npix*(np.fft.fft2(a)*np.fft.fft2(b).conj()).real
    return np.array([power[m].mean() for lo,hi,m in bands])
cxx=spectrum(X,X)
cgg=np.array([spectrum(g,g) for g in gmaps])
cxg=np.array([spectrum(X,g) for g in gmaps])
shot=theta**2/counts
assert np.all(cxx>=0) and np.all(cgg>=0)
assert np.all(cxg**2<=cxx[None,:]*cgg*(1+1e-10)+1e-30)
fig,axes=plt.subplots(1,3,figsize=(15,4.6),constrained_layout=True)
colors=plt.get_cmap('tab10')
for i in range(n_target):
    label=f'{target_edges[i]:g}<z<{target_edges[i+1]:g}'
    axes[0].plot(ell,cgg[i],'-o',color=colors(i),label=label,ms=4)
    axes[0].plot(ell,cgg[i]-shot[i],':',color=colors(i),lw=1.5)
    axes[2].plot(ell,cxg[i],'-o',color=colors(i),label=label,ms=4)
axes[1].plot(ell,cxx,'-o',color='black',ms=4)
axes[1].set_yscale('log')
for ax,values in [(axes[0],np.r_[cgg.ravel(),(cgg-shot[:,None]).ravel()]),(axes[2],cxg.ravel())]:
    ax.set_yscale('symlog',linthresh=max(np.max(abs(values))*0.01,1e-30))
    ax.axhline(0,color='0.6',lw=0.6);ax.legend(fontsize=9)
for ax in axes: ax.set_xscale('log');ax.set_xlabel(r'Multipole $L$')
axes[0].text(0.03,0.03,'Solid: raw; dotted: shot-subtracted',transform=axes[0].transAxes,fontsize=8,
             bbox=dict(facecolor='white',alpha=0.8,edgecolor='none'))
axes[0].set(title=r'$C_L^{g_i g_i}$',ylabel=r'$C_L$ [sr]')
axes[1].set(title=r'$C_L^{XX}$, full cone',ylabel=r'$C_L$ [rad$^4$ m$^{-8}$ sr]')
axes[2].set(title=r'$C_L^{Xg_i}$',ylabel=r'$C_L$ [rad$^2$ m$^{-4}$ sr]')
fig.suptitle(label_prefix+f'Fiducial spectra; {fov*fov:g} deg² patch; unfiltered theory')
save_figure(fig,'06_fiducial_angular_spectra')
rows=[]
for i in range(n_target):
    for b,(lo,hi,m) in enumerate(bands):
        rows.append(dict(z_lo=target_edges[i],z_hi=target_edges[i+1],L=ell[b],L_lo=lo,L_hi=hi,
            patch_modes=int(m.sum()),N_gal=counts[i],n_deg2=counts[i]/fov**2,
            C_gg_sr=cgg[i,b],N_gg_sr=shot[i],C_gg_minus_shot_sr=cgg[i,b]-shot[i],
            C_XX_rad4_m8_sr=cxx[b],C_Xg_rad2_m4_sr=cxg[i,b],
            r_Xg=cxg[i,b]/np.sqrt(cxx[b]*cgg[i,b]) if cxx[b]*cgg[i,b]>0 else np.nan))
spectra_table=pd.DataFrame(rows)
spectra_table.to_csv(OUTPUT_DIR/'fiducial_spectra.csv',index=False)
display(spectra_table.head())

## 7. 保存・理論的収束

全gas/group chunksを使うが、TNGの数値分解能・NGP格子・視線step・snapshot cadenceは有限。
`RM_GRID_N`、`THEORY_MAP_N`、`LOS_STEP_IN_GRID_CELLS` を変えてRMとスペクトルの収束を検証する。
star-forming gasのelectron abundanceはeffective ISMモデル依存なので、含む/除く比較にも意味がある。
全star-bearing subhalosのsampleは数値的な質量完全性を保証しない。閾値を設定した比較も行える。
TNG300-3で全セルを使ってもTNG300-1の情報にはならない。必要ならgeometryを同じboxの別runへ移して検証する。

gas磁場投影は質量重み付き、RMはelectron-weightedであり、captionで区別する。
観測beam・雑音・サーベイ選択・photo-z誤差はこのnotebookでは使用しない。

In [ ]:
assert np.isclose(X.mean(),0,atol=1e-10*max(1.,np.std(X)))
assert np.allclose(gmaps.mean(axis=(1,2)),0,atol=1e-12)
provenance=dict(source_directory=str(MAIN_OUTPUT_DIR.resolve()),source_hashes=input_hashes,
    main_configuration=config,status='fiducial finite-patch figures; not an observational forecast',
    galaxy_selection=dict(true_z=True,min_stellar_mass=GALAXY_MIN_STELLAR_MASS_MSUN,
        min_star_particles=GALAXY_MIN_STAR_PARTICLES,valid_subhalo_flag=True,thinning=False),
    spectrum_galaxy_smoothing='none',display_smoothing_arcmin=0,rm_smoothing_arcmin=0,
    rm_grid_n=grid_n,theory_map_n=n,los_step_in_grid_cells=LOS_STEP_IN_GRID_CELLS,
    theory_cone_metadata=cone_metadata,theory_cone_cache=str(cone_cache),
    gas_cache=str(gas_cache) if gas_cache is not None else None,gas_metadata=gas_metadata,
    gas_projection='NGP full box by default; mass-weighted B rms and signed LOS mean',
    shot_noise='1/n_sr; raw and subtracted gg both exported',actual_patch_area_deg2=fov*fov,
    versions={k:__import__(k).__version__ for k in ['numpy','scipy','pandas','matplotlib','h5py','astropy']})
(OUTPUT_DIR/'figure_configuration.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
print('Figures and data:',OUTPUT_DIR.resolve())
display(pd.DataFrame([dict(file=p.name,KiB=round(p.stat().st_size/1024,1))
                      for p in sorted(OUTPUT_DIR.iterdir()) if p.is_file()]))